In [27]:
import pandas as pd

# Load the CSV file into a DataFrame
# Replace 'your_file_name.csv' with your file's path
df = pd.read_csv('stock_prices.csv')

# Display the first 5 rows to check
print(df.head())

         Date       AAPL       AMZN       GOOG       MSFT
0  2018-01-02  40.380989  59.450500  52.888073  79.198318
1  2018-01-03  40.373955  60.209999  53.756138  79.566910
2  2018-01-04  40.561493  60.479500  53.950802  80.267189
3  2018-01-05  41.023300  61.457001  54.736919  81.262383
4  2018-01-08  40.870937  62.343498  54.970818  81.345314


In [28]:
# Compute log10(a_n / a_{n-1}) for all columns except the first
import numpy as np

# Treat the first column as an identifier (e.g. date/index) and skip it
cols = df.columns[1:]

# Compute ratios and take log10; this updates only the selected columns
df.loc[:, cols] = np.log10(df[cols] / df[cols].shift(1))

# Drop the first row which will be NaN due to the shift and reset the index
df = df.dropna().reset_index(drop=True)

#"squash" using tanh
df.loc[:, cols] = np.arctan(df[cols])

# Show the first few transformed rows
print(df.head())

         Date      AAPL      AMZN      GOOG      MSFT
0  2018-01-03 -0.000076  0.005513  0.007070  0.002017
1  2018-01-04  0.002013  0.001940  0.001570  0.003806
2  2018-01-05  0.004917  0.006963  0.006282  0.005351
3  2018-01-08 -0.001616  0.006220  0.001852  0.000443
4  2018-01-09 -0.000050  0.002026 -0.000267 -0.000295


In [52]:
#split into windows
WINDOW_SIZE = 4 #time windows of 4

all_features = []
all_raw_labels = []
    
    
for col in df.columns[1:]:
        
    stock_series = df[col]
    feature_list = []
    label_list = []
        
    # Slide a window across this one stock's time series
    # We stop (window_size - 1) from the end
    for i in range(len(stock_series) - WINDOW_SIZE + 1):
            
        # The full window (e.g., 4 log-returns)
        window = stock_series.iloc[i : i + WINDOW_SIZE]
            
        # Features are the first N-1 (e.g., 3)
        features = window[:-1].values
            
        # Label is the last one (e.g., the 4th)
        label = window.iloc[-1]
            
        # Get the date of the label
        date = stock_series.index[i + WINDOW_SIZE - 1]

        feature_list.append(features)
        label_list.append(label)
            
    all_features.append(feature_list)
    all_raw_labels.append(label_list)

all_features = np.array(all_features)
all_raw_labels = np.array(all_raw_labels)

print(np.shape(all_features), all_features[:,:5])
print(np.shape(all_raw_labels), all_raw_labels[:,:5])

(4, 1505, 3) [[[-7.56599731e-05  2.01263650e-03  4.91662378e-03]
  [ 2.01263650e-03  4.91662378e-03 -1.61599532e-03]
  [ 4.91662378e-03 -1.61599532e-03 -4.98608678e-05]
  [-1.61599532e-03 -4.98608678e-05 -9.97794609e-05]
  [-4.98608678e-05 -9.97794609e-05  2.45997138e-03]]

 [[ 5.51304964e-03  1.93956773e-03  6.96305839e-03]
  [ 1.93956773e-03  6.96305839e-03  6.21972594e-03]
  [ 6.96305839e-03  6.21972594e-03  2.02589951e-03]
  [ 6.21972594e-03  2.02589951e-03  5.64738986e-04]
  [ 2.02589951e-03  5.64738986e-04  7.67009720e-03]]

 [[ 7.07019780e-03  1.56984359e-03  6.28236518e-03]
  [ 1.56984359e-03  6.28236518e-03  1.85184240e-03]
  [ 6.28236518e-03  1.85184240e-03 -2.66832013e-04]
  [ 1.85184240e-03 -2.66832013e-04 -1.43527805e-03]
  [-2.66832013e-04 -1.43527805e-03  1.14468505e-03]]

 [[ 2.01652772e-03  3.80554447e-03  5.35144588e-03]
  [ 3.80554447e-03  5.35144588e-03  4.42988892e-04]
  [ 5.35144588e-03  4.42988892e-04 -2.95370834e-04]
  [ 4.42988892e-04 -2.95370834e-04 -1.9736044

In [53]:
# Convert labels to bins
N_BINS = 2 #increasing or decreasing

all_labels = []

# if N_BINS > 2, we would average max and min. But we don't need to do that
for stock_raw_labels in all_raw_labels:
    stock_labels = []

    min_label = np.min(stock_raw_labels)
    max_label = np.max(stock_raw_labels)
    bin_edges = np.linspace(min_label, max_label, N_BINS + 1)
    bin_edges = bin_edges[:-1]  # Remove last edge

    for raw_label in stock_raw_labels:
        # Find the bin index for this raw label
        bin_index = np.digitize(raw_label, bin_edges) - 1  # digitize returns 1-based index
        # Clip to ensure it falls within [0, N_BINS-1]
        bin_index = min(max(bin_index, 0), N_BINS - 1)
        stock_labels.append(bin_index)

    all_labels.append(stock_labels)

all_labels = np.array(all_labels)
print(np.shape(all_labels), all_labels[:,:30])

    

(4, 1505) [[1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 0 0 1 0 0 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 0 0 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 0 1 0 0 1 1 1 1 1 1 1]
 [1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0 1 1 1 1 1 1 1]]


In [54]:
# split into train and test sets

TRAIN_RATIO = 0.8

split_index = int(all_features.shape[1] * TRAIN_RATIO)

train_features = all_features[:, :split_index, :]
train_labels = all_labels[:, :split_index]

test_features = all_features[:, split_index:, :]
test_labels = all_labels[:, split_index:]

print("Train features shape:", train_features.shape)
print("Train labels shape:", train_labels.shape)

print("Test features shape:", test_features.shape)
print("Test labels shape:", test_labels.shape)


Train features shape: (4, 1204, 3)
Train labels shape: (4, 1204)
Test features shape: (4, 301, 3)
Test labels shape: (4, 301)


In [65]:
import pennylane as qml

# 0-1: context qubits
# 2-4: input qubits
# 5: output qubit
n_context_qb = np.log2(train_features.shape[0])  # Number of stocks
n_input_qb = train_features.shape[2]  # Should be 3
n_output_qb = np.log2(N_BINS)

dev = qml.device("default.qubit", wires=6)


for stock_features in train_features: 

    for features in stock_features[:1]: # only first element for testing

        @qml.qnode(dev)
        def circuit(features):
            # Encode input features into qubits 2, 3, 4
            for i in range(n_input_qb):
                qml.RY(features[i], wires=i + n_context_qb)

            #TODO: Implement shared variational layers

            #TODO: Implement specify by context layers

            # Measure output qubit (wire 5)
            return qml.probs(wires=n_context_qb + n_input_qb)
        
        # Execute the circuit
        result = circuit(features)
        print("Features:", features, "Output expectation:", result)

        drawing = qml.draw(circuit)(features)
        print(drawing)


Features: [-7.56599731e-05  2.01263650e-03  4.91662378e-03] Output expectation: [1. 0.]
2: ──RY(-0.00)─┤       
3: ──RY(0.00)──┤       
4: ──RY(0.00)──┤       
5: ────────────┤  Probs
Features: [0.00551305 0.00193957 0.00696306] Output expectation: [1. 0.]
2: ──RY(0.01)─┤       
3: ──RY(0.00)─┤       
4: ──RY(0.01)─┤       
5: ───────────┤  Probs
Features: [0.0070702  0.00156984 0.00628237] Output expectation: [1. 0.]
2: ──RY(0.01)─┤       
3: ──RY(0.00)─┤       
4: ──RY(0.01)─┤       
5: ───────────┤  Probs
Features: [0.00201653 0.00380554 0.00535145] Output expectation: [1. 0.]
2: ──RY(0.00)─┤       
3: ──RY(0.00)─┤       
4: ──RY(0.01)─┤       
5: ───────────┤  Probs
